# HypeClipper v4 - 1080p Hook & Payoff Shorts + Top 10 Viral
Generates **25s-50s** crisp (**1080p+**) vertical Shorts with word-synced captions, stereo hook SFX, custom 9:16 layouts, and a `top_10_viral/` folder with thumbnails.
1. **Runtime > Change runtime type > T4 GPU**.
2. Run cells top to bottom; outputs save to `MyDrive/HypeClipper/`.

In [ ]:
#@title 1. Check GPU & Mount Google Drive
import shutil, subprocess, os
from google.colab import drive

if shutil.which('nvidia-smi'):
    gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
    print('GPU:', gpu.stdout.strip())
else:
    print('No GPU detected; running on CPU.')

drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/HypeClipper', exist_ok=True)

In [ ]:
#@title 2. Install HypeClipper Engine, Fonts & Dependencies
import os, subprocess

subprocess.run(['apt-get', '-qq', 'update'], check=False)
subprocess.run(['apt-get', '-qq', 'install', '-y', 'ffmpeg', 'fonts-dejavu-core', 'fontconfig'], check=False)
subprocess.run(['fc-cache', '-f'], check=False)

!curl -fsSL https://deno.land/install.sh | sh > /dev/null 2>&1 || true
os.environ['PATH'] += ':/root/.deno/bin:/usr/local/bin'
!rm -rf hype-clipper && git clone https://github.com/Aromalajith8/hype-clipper.git hype-clipper
!pip install -q -r hype-clipper/requirements.txt
print('HypeClipper v4 ready!')

In [ ]:
#@title 3. Stream Source & Free AI Setup
source = '' #@param {type:"string"}
cookies_path = '/content/drive/MyDrive/cookies.txt' #@param {type:"string"}
ai_provider = 'auto' #@param ['auto', 'groq', 'gemini', 'ollama', 'heuristic']
ollama_model = 'qwen3.5:9b' #@param ['qwen3.5:9b', 'qwen3:8b', 'qwen2.5:7b']
groq_api_key = '' #@param {type:"string"}
gemini_api_key = '' #@param {type:"string"}

In [ ]:
#@title 4. Preview Styles & Configure Layout, Facecam, SFX, Clarity
from IPython.display import HTML, display
layout = 'cam-top' #@param ['blur-bg', 'crop-focus', 'cam-top', 'cam-bottom']
cam_pos = 'top-left' #@param ['top-left', 'top-right', 'bottom-left', 'bottom-right']
add_sfx = True #@param {type:"boolean"}
min_height = 1080 #@param [1080, 1440, 2160]
min_len = 25 #@param {type:"slider", min:20, max:40, step:5}
max_len = 50 #@param {type:"slider", min:25, max:50, step:5}
max_clips = 40 #@param {type:"slider", min:5, max:80, step:5}
caption_style = 'bold-pop' #@param ['auto', 'bold-pop', 'beast-bold', 'karaoke-highlight', 'word-by-word', 'two-line-stack', 'highlight-box', 'neon-glow', 'impact-shake', 'emoji-pop', 'typewriter', 'bounce', 'gradient-stroke', 'comic-burst', 'minimal-clean', 'gaming-glitch', 'none']
hook_style = 'bold-box' #@param ['bold-box', 'outline-glow', 'top-bar', 'none']
display(HTML('\n<div style="background:#0d0f14;padding:14px;border-radius:10px;border:1px solid #262b38;font-family:sans-serif;color:#eee;">\n  <div style="font-weight:700;margin-bottom:10px;font-size:15px;">Caption & Hook Style Visual Preview (pick in Step 4 below)</div>\n  <div style="display:grid;grid-template-columns:repeat(3,1fr);gap:10px;text-align:center;">\n    <div style="background:#050608;padding:14px;border-radius:6px;"><div style="font-weight:900;font-size:20px;color:#fff;">NO <span style="color:#ff0;">WAY</span></div><div style="font-size:11px;color:#8a93a6;margin-top:6px;">bold-pop</div></div>\n    <div style="background:#050608;padding:14px;border-radius:6px;"><div style="font-family:Impact,sans-serif;font-size:22px;color:#fff;">LET\'S <span style="color:#3f3;">GO</span></div><div style="font-size:11px;color:#8a93a6;margin-top:6px;">beast-bold</div></div>\n    <div style="background:#050608;padding:14px;border-radius:6px;"><div style="font-weight:800;font-size:16px;color:#fff;">HE IS <span style="color:#ff0;">CHEATING</span></div><div style="font-size:11px;color:#8a93a6;margin-top:6px;">karaoke-highlight</div></div>\n    <div style="background:#050608;padding:14px;border-radius:6px;"><div style="font-family:Impact,sans-serif;font-size:24px;color:#fff;">INSANE</div><div style="font-size:11px;color:#8a93a6;margin-top:6px;">word-by-word</div></div>\n    <div style="background:#050608;padding:14px;border-radius:6px;"><div style="font-weight:800;font-size:15px;line-height:1.2;color:#fff;">I CAN\'T<br>BELIEVE <span style="color:#ff0;">THIS</span></div><div style="font-size:11px;color:#8a93a6;margin-top:6px;">two-line-stack</div></div>\n    <div style="background:#050608;padding:14px;border-radius:6px;"><div style="font-weight:900;font-size:17px;"><span style="background:#ff0;color:#000;padding:2px 6px;">CLIP THAT</span></div><div style="font-size:11px;color:#8a93a6;margin-top:6px;">highlight-box</div></div>\n    <div style="background:#050608;padding:14px;border-radius:6px;"><div style="font-weight:900;font-size:18px;color:#fff;text-shadow:0 0 8px #0ff;">CHAT <span style="color:#0ff;">LOOK</span></div><div style="font-size:11px;color:#8a93a6;margin-top:6px;">neon-glow</div></div>\n    <div style="background:#050608;padding:14px;border-radius:6px;"><div style="font-family:Impact,sans-serif;font-size:22px;color:#ff0;">BRO!</div><div style="font-size:11px;color:#8a93a6;margin-top:6px;">impact-shake</div></div>\n    <div style="background:#050608;padding:14px;border-radius:6px;"><div style="font-family:monospace;font-weight:700;font-size:16px;color:#fff;">WAIT FOR <span style="color:#ff0;">IT</span></div><div style="font-size:11px;color:#8a93a6;margin-top:6px;">typewriter</div></div>\n    <div style="background:#050608;padding:14px;border-radius:6px;"><div style="font-weight:900;font-size:19px;color:#ff0;-webkit-text-stroke:1px #f3c;">CLUTCH</div><div style="font-size:11px;color:#8a93a6;margin-top:6px;">gradient-stroke</div></div>\n    <div style="background:#050608;padding:14px;border-radius:6px;"><div style="font-family:Impact,sans-serif;font-size:21px;color:#f44;">BOOM</div><div style="font-size:11px;color:#8a93a6;margin-top:6px;">comic-burst</div></div>\n    <div style="background:#050608;padding:14px;border-radius:6px;"><div style="font-size:15px;color:#fff;">That Was <span style="color:#ff0;">Close</span></div><div style="font-size:11px;color:#8a93a6;margin-top:6px;">minimal-clean</div></div>\n    <div style="background:#050608;padding:14px;border-radius:6px;"><div style="background:#ff0;color:#000;font-weight:900;font-size:13px;padding:4px 6px;display:inline-block;">HE DIDN\'T SEE THAT</div><div style="font-size:11px;color:#8a93a6;margin-top:6px;">bold-box hook</div></div>\n    <div style="background:#050608;padding:14px;border-radius:6px;"><div style="font-weight:900;font-size:14px;color:#fff;text-shadow:0 0 8px #ff0;">WAIT FOR THE END</div><div style="font-size:11px;color:#8a93a6;margin-top:6px;">outline-glow hook</div></div>\n    <div style="background:#050608;padding:14px;border-radius:6px;"><div style="background:#e22;color:#fff;font-weight:900;font-size:13px;padding:4px 8px;">CHAT WENT CRAZY</div><div style="font-size:11px;color:#8a93a6;margin-top:6px;">top-bar hook</div></div>\n  </div>\n</div>\n'))
print(f'Selected -> layout={layout}, cam_pos={cam_pos}, caption={caption_style}, hook={hook_style}, sfx={add_sfx}')

In [ ]:
#@title 5. Run HypeClipper v4 Generator
import glob, os, shutil, subprocess, time

assert source.strip(), 'Paste a stream URL or Drive video path in Step 3 first!'

if groq_api_key.strip():
    os.environ['GROQ_API_KEY'] = groq_api_key.strip()
if gemini_api_key.strip():
    os.environ['GEMINI_API_KEY'] = gemini_api_key.strip()

# Only download/start Ollama if user explicitly selects ollama (prevents wasting ~5 min on auto)
if ai_provider == 'ollama':
    if not shutil.which('ollama'):
        subprocess.run(['apt-get', '-qq', 'install', '-y', 'zstd', 'curl'], check=False)
        subprocess.run('curl -fsSL -L https://github.com/ollama/ollama/releases/latest/download/ollama-linux-amd64.tgz -o /tmp/ollama.tgz && mkdir -p /usr/local && tar -xzf /tmp/ollama.tgz -C /usr/local && rm -f /tmp/ollama.tgz', shell=True, check=False)
    ollama_bin = shutil.which('ollama') or '/usr/local/bin/ollama'
    if os.path.exists(ollama_bin):
        subprocess.Popen([ollama_bin, 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        time.sleep(3)
        subprocess.run([ollama_bin, 'pull', ollama_model], check=False)
elif ai_provider == 'auto' and not groq_api_key.strip() and not gemini_api_key.strip():
    print('No API key set -> using fast local Hook + Energy scorer (add a free Groq/Gemini key in Step 3 for LLM ranking).')

cmd = [
    'python3', 'hype-clipper/clipper.py', source.strip(),
    '--clips', str(max_clips),
    '--min-len', str(min_len),
    '--max-len', str(min(max_len, 50)),
    '--min-height', str(max(1080, min_height)),
    '--layout', layout,
    '--cam-pos', cam_pos,
    '--caption-style', caption_style,
    '--hook-style', hook_style,
    '--out', '/content/drive/MyDrive/HypeClipper',
    '--ai-provider', ai_provider,
    '--ollama-model', ollama_model,
]

if add_sfx:
    cmd.append('--sfx')
if cookies_path.strip() and os.path.exists(cookies_path.strip()):
    cmd += ['--cookies', cookies_path.strip()]

subprocess.run(cmd, check=False)
latest_sessions = sorted(glob.glob('/content/drive/MyDrive/HypeClipper/*'), key=os.path.getmtime)
LAST_SESSION_DIR = latest_sessions[-1] if latest_sessions else None
print('Latest HypeClipper session:', LAST_SESSION_DIR)

In [ ]:
#@title 6. Preview Top Viral Clips & Thumbnails
import glob, os
from IPython.display import Image, Video, display

sess = globals().get('LAST_SESSION_DIR')
if not sess or not os.path.isdir(sess):
    sessions = sorted(glob.glob('/content/drive/MyDrive/HypeClipper/*'), key=os.path.getmtime)
    sess = sessions[-1] if sessions else None

if sess:
    print('Session:', sess)
    print('Top 10 Viral Folder:', os.path.join(sess, 'top_10_viral'))
    mp4s = sorted(glob.glob(os.path.join(sess, 'top_10_viral', '*.mp4')))[:3]
    for m in mp4s:
        print('Top Viral Preview:', os.path.basename(m))
        display(Video(m, embed=True, width=320))
else:
    print('No completed session found yet. Run Step 5 first.')